# 04 · Entrenamiento de los 3 modelos en Google Colab (GPU)

Entrena `mobilenet_lstm`, `resnet_lstm` y `mobilenet_baseline` sobre RWF-2000 con
`src/models/train.py`, igual que en local pero con GPU.

Particion (ver docstring de `train.py`):
- `train/` se divide 85/15 **por video fuente**: 85 % entrena, 15 % es el val interno (mejor checkpoint y early stopping).
- `val/` entero es el **test**: se evalua una sola vez al final con el mejor checkpoint. Esas metricas van a `artifacts/metrics_<modelo>.json` (las del informe).

## Antes de correr
1. **Entorno > Cambiar tipo de entorno de ejecucion > GPU T4.**
2. Subir a la raiz de tu Drive el zip del repo (`security_repo.zip`, generado con `git archive`).
3. Tener el dataset en Drive. Si la carpeta `datasets/` del grupo esta en *Compartidos conmigo*, agregarle un acceso directo a *Mi unidad* y ajustar `RWF_DRIVE` abajo.
4. No cerrar la pestana: Colab gratis corta la sesion si queda inactiva.

Cada modelo copia su log, sus metricas y su `.pt` a `OUT_DRIVE` apenas termina, asi una desconexion no borra lo ya entrenado.

In [ ]:
# Confirmar que hay GPU. Si falla, cambiar el tipo de entorno a GPU T4.
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Rutas en Drive: ajustarlas si tu carpeta tiene otro nombre.
REPO_ZIP = '/content/drive/MyDrive/security_repo.zip'
RWF_DRIVE = '/content/drive/MyDrive/datasets/rwf2000'
OUT_DRIVE = '/content/drive/MyDrive/security_artifacts'

FECHA = '2026-10-29'
EPOCHS = 15
# Colab gratis trae 2 CPU: 2 procesos decodifican videos mientras la GPU entrena.
NUM_WORKERS = 2

In [ ]:
# Codigo del repo: se descomprime el zip en /content/Security.
!rm -rf /content/Security && mkdir -p /content/Security
!unzip -q "{REPO_ZIP}" -d /content/Security
%cd /content/Security
!ls

In [ ]:
# Dataset al disco local de Colab: leer 2000 .avi directo desde Drive en cada
# epoca es varias veces mas lento que copiarlos una vez (tarda ~5-10 min).
import pathlib

!mkdir -p /content/rwf2000
!cp -r "{RWF_DRIVE}"/. /content/rwf2000/

# Si en Drive esta como .zip (o varios), se descomprimen aca.
for z in sorted(pathlib.Path('/content/rwf2000').glob('*.zip')):
    print('Descomprimiendo', z.name)
    !unzip -q -o "{z}" -d /content/rwf2000

# La raiz del dataset es la carpeta que tiene train/ y val/ adentro.
roots = [p.parent for p in pathlib.Path('/content/rwf2000').rglob('train')
         if p.is_dir() and (p.parent / 'val').is_dir()]
assert roots, 'No se encontro una carpeta con train/ y val/ dentro de /content/rwf2000'
DATA = str(roots[0])
print('Dataset en', DATA)

# Se esperan 800 / 800 / 200 / 200 (data/README.md).
for d in sorted(pathlib.Path(DATA).glob('*/*')):
    if d.is_dir():
        print(f'{d.parent.name}/{d.name}: {len(list(d.glob("*.avi")))} videos')

In [ ]:
# Colab ya trae torch con CUDA, torchvision, OpenCV, scikit-learn y tqdm.
# No se instala requirements.txt: fija torch==2.3.1 (CPU en local) y
# reinstalarlo aca tarda y puede romper la version con CUDA.
import torch, torchvision, cv2, sklearn, tqdm, sys
print('python', sys.version.split()[0])
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('torchvision', torchvision.__version__, '| opencv', cv2.__version__,
      '| sklearn', sklearn.__version__, '| tqdm', tqdm.__version__)

In [ ]:
# Smoke test (~1 min): 10 train + 5 val + 5 test, 1 epoca. Si esto falla, no
# vale la pena lanzar el entrenamiento largo.
!python -m src.models.train --model mobilenet_lstm --dataset-path "{DATA}" --smoke-test --output-dir /content/smoke

In [ ]:
import os

# Barra de progreso cada 30 s: si no, el log se llena de lineas de tqdm.
os.environ['TQDM_MININTERVAL'] = '30'
os.makedirs(OUT_DRIVE, exist_ok=True)


def entrenar(modelo: str) -> None:
    """Entrena un modelo, guarda el log y copia los resultados a Drive."""
    log = f'artifacts/train_{modelo}_{FECHA}.log'
    get_ipython().system(
        f'python -u -m src.models.train --model {modelo} --dataset-path "{DATA}" '
        f'--epochs {EPOCHS} --num-workers {NUM_WORKERS} 2>&1 | tee {log}'
    )
    for f in (log, f'artifacts/metrics_{modelo}.json',
              f'artifacts/metrics_{modelo}_internal_val.json',
              f'artifacts/model_{modelo}.pt'):
        if os.path.exists(f):
            get_ipython().system(f'cp "{f}" "{OUT_DRIVE}/"')
    print('Copiado a', OUT_DRIVE)

## Entrenamiento
Primero `mobilenet_lstm`, que es el que carga la API. Cada celda se puede correr sola si la sesion se corta.

In [ ]:
entrenar('mobilenet_lstm')

In [ ]:
entrenar('resnet_lstm')

In [ ]:
entrenar('mobilenet_baseline')

## Resumen (metricas de TEST)
Bajar de `OUT_DRIVE` los `metrics_*.json` y `train_*.log` a `artifacts/` del repo. Los `.pt` no se commitean.

In [ ]:
import json

print(f"{'modelo':<20}{'acc':>8}{'f1_macro':>10}{'recall_sosp':>13}{'mejor_epoca':>13}{'ms/clip':>9}")
for modelo in ('mobilenet_baseline', 'mobilenet_lstm', 'resnet_lstm'):
    path = f'artifacts/metrics_{modelo}.json'
    if not os.path.exists(path):
        print(f'{modelo:<20}  (sin metricas)')
        continue
    r = json.load(open(path))
    final = r['final']
    recall = final['recall_per_class'][r['class_names'][1]]
    print(f"{modelo:<20}{final['accuracy']:>8.3f}{final['f1_macro']:>10.3f}"
          f"{recall:>13.3f}{r['best_epoch']:>13}{r['inference_ms']:>9}")